In [1]:
import os
from typing import TypedDict,Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph,END


C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
API_KEY = 'API_KEY'

In [3]:
llm = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    google_api_key = API_KEY,
    temperature = 0.2
)

In [4]:
class SimActivationState(TypedDict):
    customer_id:str
    customer_name:str
    aadhar_number:str
    pan_number:str
    location:str
    previous_sim_requests:str
    document_status:str

    kyc_status:str
    fraud_risk:str
    fraud_reasoning:str
    activation_decision:str
    final_message:str

In [5]:
@tool
def verify_kyc(aadhar_number:str,pan_number:str,document_status:str) -> str:
    '''
        Verify KYC Details
    '''
    aadhar_valid = aadhar_number.isdigit() and len(aadhar_number) == 12
    pan_valid = (
        len(pan_number) == 10
        and pan_number[:5].isalpha()
        and pan_number[5:9].isdigit()
        and pan_number[9].isalpha()
    )

    if document_status == "PENDING":
        return "PENDING_VERIFICATION"

    if aadhar_valid and pan_valid:
        return "VERIFIED"

    return "INVALID_DOCUMENT"

In [6]:
def  kyc_verification_node(state:SimActivationState) -> SimActivationState:
    kyc_result = verify_kyc.invoke({
        "aadhar_number": state["aadhar_number"],
        "pan_number": state["pan_number"],
        "document_status": state["document_status"]
    })

    return {**state,"kyc_status": kyc_result}

In [7]:
def fraud_detection_node(state: SimActivationState) -> SimActivationState:
    system_prompt = (
        "You are a telecom fraud detection AI. "
        "Analyze the customer profile and classify the fraud risk. "
        "Reply with EXACTLY one of: LOW_RISK, MEDIUM_RISK, HIGH_RISK "
        "followed by a newline and a one-sentence explanation."
    )
 
    user_prompt = (
        f"Customer Profile:\n"
        f"- Name: {state['customer_name']}\n"
        f"- Location: {state['location']}\n"
        f"- Previous SIM requests in last 30 days: {state['previous_sim_requests']}\n"
        f"- KYC Status: {state['kyc_status']}\n\n"
        "Classify the fraud risk level."
    )
 
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ])
 
    raw = response.content.strip()
    lines = raw.split("\n", 1)
    fraud_risk = lines[0].strip()
    reasoning  = lines[1].strip() if len(lines) > 1 else "No additional reasoning."
 
    for label in ["HIGH_RISK", "MEDIUM_RISK", "LOW_RISK"]:
        if label in fraud_risk.upper():
            fraud_risk = label
            break
 
    print(f"Fraud Risk : {fraud_risk}")
    print(f"Reasoning : {reasoning}")
 
    return {**state, "fraud_risk": fraud_risk, "fraud_reasoning": reasoning}

In [8]:
def activation_decision_node(state: SimActivationState) -> SimActivationState:
    decision_map = {
        "LOW_RISK": ("SIM Activated","SIM successfully activated for the customer."),
        "MEDIUM_RISK": ("Manual Review","Application flagged for manual agent review."),
        "HIGH_RISK": ("Reject Application","Application rejected due to high fraud risk."),
    }
 
    decision, message = decision_map.get(
        state["fraud_risk"],
        ("Manual Review", "Unknown risk level — routed to manual review.")
    )
 
    print(f"Decision : {decision}")
    print(f"Message : {message}")
 
    return {**state, "activation_decision": decision, "final_message": message}

In [9]:
def build_sim_workflow() -> StateGraph:
    graph = StateGraph(SimActivationState)

    graph.add_node("kyc_verification",kyc_verification_node)
    graph.add_node("fraud_detection",fraud_detection_node)
    graph.add_node("activation_decision",activation_decision_node)

    graph.set_entry_point("kyc_verification")
    graph.add_edge("kyc_verification","fraud_detection")
    graph.add_edge("fraud_detection","activation_decision")
    graph.add_edge("activation_decision", END)

    return graph.compile()

In [10]:
def run_sim_workflow(customer: dict) -> SimActivationState:
    workflow = build_sim_workflow()
    result = workflow.invoke(customer)
    print(f"\nFINAL SUMMARY")
    print(f"Customer : {result['customer_name']}")
    print(f"KYC : {result['kyc_status']}")
    print(f"Fraud Risk: {result['fraud_risk']}")
    print(f"Decision : {result['activation_decision']}")
    print(f"Message : {result['final_message']}")
    return result

In [11]:
if __name__ == "__main__":
 
    test_customers = [
        # --- Case A: Clean customer, Low Risk ---
        {
            "customer_id": "CUST001",
            "customer_name": "Priya Sharma",
            "aadhar_number": "234567890123",
            "pan_number": "ABCDE1234F",
            "location": "Mumbai, Maharashtra",
            "previous_sim_requests":  1,
            "document_status": "OK",
            "kyc_status": "",
            "fraud_risk":"",
            "fraud_reasoning": "",
            "activation_decision":"",
            "final_message":"",
        },
        # --- Case B: Multiple requests, Medium/High Risk ---
        {
            "customer_id": "CUST002",
            "customer_name": "Rahul Verma",
            "aadhar_number":"987654321098",
            "pan_number": "XYZAB5678G",
            "location": "Jaipur, Rajasthan",
            "previous_sim_requests": 8,
            "document_status": "OK",
            "kyc_status": "",
            "fraud_risk": "",
            "fraud_reasoning":"",
            "activation_decision":"",
            "final_message": "",
        },
        # --- Case C: Invalid documents ---
        {
            "customer_id":"CUST003",
            "customer_name": "Amit Kumar",
            "aadhar_number": "12345",
            "pan_number": "INVALID", 
            "location": "Delhi",
            "previous_sim_requests": 2,
            "document_status": "OK",
            "kyc_status": "",
            "fraud_risk":"",
            "fraud_reasoning":"",
            "activation_decision":"",
            "final_message":"",
        },
    ]
 
    for customer in test_customers:
        run_sim_workflow(customer)
        print()

Fraud Risk : LOW_RISK
Reasoning : The customer has a verified KYC status and only one previous SIM request in the last 30 days, which are not indicators of high fraud risk.
Decision : SIM Activated
Message : SIM successfully activated for the customer.

FINAL SUMMARY
Customer : Priya Sharma
KYC : VERIFIED
Fraud Risk: LOW_RISK
Decision : SIM Activated
Message : SIM successfully activated for the customer.

Fraud Risk : HIGH_RISK
Reasoning : The unusually high number of SIM requests within a 30-day period strongly suggests potential fraudulent activity, despite the verified KYC status.
Decision : Reject Application
Message : Application rejected due to high fraud risk.

FINAL SUMMARY
Customer : Rahul Verma
KYC : VERIFIED
Fraud Risk: HIGH_RISK
Decision : Reject Application
Message : Application rejected due to high fraud risk.

Fraud Risk : HIGH_RISK
Reasoning : The invalid KYC document status combined with multiple recent SIM requests strongly indicates a high risk of fraudulent activity